In [2]:
import os

# Check current working directory
print("Current directory:", os.getcwd())

# List everything in it
print("Files here:", os.listdir())

# Common Colab locations to check
for path in ["/content", "/content/sample_data", "/content/drive/MyDrive"]:
    if os.path.exists(path):
        print(f"\n{path} contains:")
        print(os.listdir(path))

Current directory: C:\Users\mehvish shaikh\OneDrive\Documents\Thesis
Files here: ['.ipynb_checkpoints', '1781820278.2604444.mummychog_results', 'ABSOLUTE_FINAL_mz_RT_HMDB_KEGG.csv', 'feces_metabolites', 'feces_metabolites.zip', 'FINAL_mtb_map_annotated.csv', 'FRANZOSA_ANNOTATION_COMPLETE.csv', 'FRANZOSA_ANNOTATION_FINAL.csv', 'FRANZOSA_HMDB_FINAL.csv', 'FRANZOSA_IBD_2019', 'FRANZOSA_IBD_2019.zip', 'FRANZOSA_KEGG_MATCHING.csv', 'FRANZOSA_MASS_SEARCH.csv', 'FRANZOSA_MATCHING_SUMMARY.csv', 'franzosa_metadata_RData.csv', 'franzosa_models_python.ipynb', 'FRANZOSA_MTB_MAP_REPRODUCED.csv', 'franzosa_preprocessing.Rmd', 'franzosa_preprocessing_python.ipynb', 'FRANZOSA_PUBCHEM_MATCHING.csv', 'franzosa_visualizations_python.ipynb', 'HMDB_100percent_matched_Franzosa.csv', 'HMDB_100percent_PPM_Franzosa.csv', 'hmdb_annotated_significant.csv', 'hmdb_annotation_1344_final.csv', 'hmdb_annotation_1344_final.png', 'hmdb_annotation_results.png', 'hmdb_annotation_table.csv', 'HMDB_Final_Fixed_Franzosa.csv

In [3]:
import os

for folder in ["FRANZOSA_IBD_2019", "iHMP_IBDMDB_2019"]:
    print(f"\n--- {folder} ---")
    print(os.listdir(folder))


--- FRANZOSA_IBD_2019 ---
['FRANZOSA_IBD_2019']

--- iHMP_IBDMDB_2019 ---
['iHMP_IBDMDB_2019']


In [4]:
import os

for folder in ["FRANZOSA_IBD_2019/FRANZOSA_IBD_2019", "iHMP_IBDMDB_2019/iHMP_IBDMDB_2019"]:
    print(f"\n--- {folder} ---")
    print(os.listdir(folder))


--- FRANZOSA_IBD_2019/FRANZOSA_IBD_2019 ---
['.RData', 'metadata.tsv', 'mtb.map.tsv', 'mtb.tsv']

--- iHMP_IBDMDB_2019/iHMP_IBDMDB_2019 ---
['.RData', 'metadata.tsv', 'mtb.map.tsv', 'mtb.tsv', 'mtb.tsv.zip']


In [5]:
import pandas as pd

metadata = pd.read_csv("FRANZOSA_IBD_2019/FRANZOSA_IBD_2019/metadata.tsv", sep="\t")
mtb = pd.read_csv("FRANZOSA_IBD_2019/FRANZOSA_IBD_2019/mtb.tsv", sep="\t")

print("Metadata shape:", metadata.shape)
print("Metabolite table shape:", mtb.shape)
print(metadata.columns.tolist())

Metadata shape: (220, 13)
Metabolite table shape: (220, 8849)
['Dataset', 'Sample', 'Subject', 'Study.Group', 'Age', 'Age.Units', 'DOI', 'Publication.Name', 'Fecal.Calprotectin', 'antibiotic', 'immunosuppressant', 'mesalamine', 'steroids']


In [6]:
df = metadata.merge(mtb, on="Sample")

print("Joined shape:", df.shape)
print(df["Study.Group"].value_counts())

Joined shape: (220, 8861)
Study.Group
CD         88
UC         76
Control    56
Name: count, dtype: int64


In [7]:
from sklearn.ensemble import IsolationForest

feature_cols = [c for c in mtb.columns if c != "Sample"]
X = df[feature_cols]
y = df["Study.Group"]

iso = IsolationForest(contamination=0.05, random_state=42)
outlier_flags = iso.fit_predict(X)

print("Samples flagged as outliers:", (outlier_flags == -1).sum())
print("Samples kept:", (outlier_flags == 1).sum())

Samples flagged as outliers: 11
Samples kept: 209


In [8]:
import numpy as np

# Apply log2 transform to the outlier-filtered data
X_filtered = X[outlier_flags == 1]
y_filtered = y[outlier_flags == 1]

X_log = np.log2(X_filtered + 1)

print("Shape after log transform:", X_log.shape)
print(X_log.iloc[:3, :5])  # peek at first 3 samples, 5 features

Shape after log transform: (209, 8848)
   C18-neg_Cluster_0001: NA  C18-neg_Cluster_0002: NA  \
0                  0.000000                 12.642054   
1                 10.676433                  4.830159   
2                  0.000000                 13.013131   

   C18-neg_Cluster_0003: NA  C18-neg_Cluster_0004: 4-hydroxystyrene  \
0                  8.178954                                9.239844   
1                  5.912679                                8.437082   
2                 12.912446                                5.064978   

   C18-neg_Cluster_0005: NA  
0                 11.227363  
1                  8.753615  
2                 12.947661  


In [9]:
from scipy.stats import kruskal

def kruskal_top_k(X, y, k):
    groups = y.unique()
    pvals = {}
    for col in X.columns:
        samples_by_group = [X.loc[y.values == g, col] for g in groups]
        try:
            stat, p = kruskal(*samples_by_group)
        except ValueError:
            p = 1.0
        pvals[col] = p
    ranked = sorted(pvals, key=pvals.get)
    return ranked[:k]

feature_set_sizes = [5, 10, 20, 40]
selected_features = {k: kruskal_top_k(X_log, y_filtered, k) for k in feature_set_sizes}

for k, feats in selected_features.items():
    print(f"\nTop {k} features:")
    print(feats)


Top 5 features:
['C18-neg_Cluster_2083: NA', 'C18-neg_Cluster_2129: NA', 'HILIC-pos_Cluster_2135: NA', 'C18-neg_Cluster_2109: NA', 'C18-neg_Cluster_2037: NA']

Top 10 features:
['C18-neg_Cluster_2083: NA', 'C18-neg_Cluster_2129: NA', 'HILIC-pos_Cluster_2135: NA', 'C18-neg_Cluster_2109: NA', 'C18-neg_Cluster_2037: NA', 'C18-neg_Cluster_2096: NA', 'C18-neg_Cluster_0450: NA', 'HILIC-neg_Cluster_1831: NA', 'C18-neg_Cluster_2021: urobilin', 'C18-neg_Cluster_0774: NA']

Top 20 features:
['C18-neg_Cluster_2083: NA', 'C18-neg_Cluster_2129: NA', 'HILIC-pos_Cluster_2135: NA', 'C18-neg_Cluster_2109: NA', 'C18-neg_Cluster_2037: NA', 'C18-neg_Cluster_2096: NA', 'C18-neg_Cluster_0450: NA', 'HILIC-neg_Cluster_1831: NA', 'C18-neg_Cluster_2021: urobilin', 'C18-neg_Cluster_0774: NA', 'C18-neg_Cluster_2128: NA', 'C18-neg_Cluster_2156: NA', 'C18-neg_Cluster_2081: NA', 'HILIC-pos_Cluster_2146: NA', 'HILIC-pos_Cluster_2231: NA', 'C18-neg_Cluster_2161: NA', 'HILIC-pos_Cluster_2093: urobilin*', 'HILIC-pos_Cl

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
import pandas as pd

le = LabelEncoder()
y_enc = le.fit_transform(y_filtered)

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

rf_grid = {"n_estimators": [200, 500], "max_features": ["sqrt", "log2"], "min_samples_leaf": [1, 3, 5]}
en_grid = {"C": [0.01, 0.1, 1, 10], "l1_ratio": [0.1, 0.5, 0.9]}

results = []

for k in [5, 10, 20, 40]:
    print(f"\n--- Running k={k} ---")
    X_k = X_log[selected_features[k]]

    # --- Random Forest ---
    rf = GridSearchCV(RandomForestClassifier(random_state=42), rf_grid, cv=inner_cv, scoring="roc_auc_ovr", n_jobs=-1)
    rf_probs = cross_val_predict(rf, X_k, y_enc, cv=outer_cv, method="predict_proba")
    rf_auc = roc_auc_score(y_enc, rf_probs, multi_class="ovr", average="macro")

    # --- Elastic Net ---
    X_k_scaled = StandardScaler().fit_transform(X_k)
    en = GridSearchCV(LogisticRegression(penalty="elasticnet", solver="saga", max_iter=5000),
                       en_grid, cv=inner_cv, scoring="roc_auc_ovr", n_jobs=-1)
    en_probs = cross_val_predict(en, X_k_scaled, y_enc, cv=outer_cv, method="predict_proba")
    en_auc = roc_auc_score(y_enc, en_probs, multi_class="ovr", average="macro")

    print(f"k={k}: RF macro-AUC={rf_auc:.3f} | ElasticNet macro-AUC={en_auc:.3f}")
    results.append({"n_features": k, "model": "RandomForest", "macro_auc": rf_auc})
    results.append({"n_features": k, "model": "ElasticNet", "macro_auc": en_auc})

results_df = pd.DataFrame(results)
results_df.to_csv("franzosa_ibdpred_style_results.csv", index=False)
print("\n", results_df)


--- Running k=5 ---
k=5: RF macro-AUC=0.715 | ElasticNet macro-AUC=0.735

--- Running k=10 ---
k=10: RF macro-AUC=0.745 | ElasticNet macro-AUC=0.751

--- Running k=20 ---
k=20: RF macro-AUC=0.760 | ElasticNet macro-AUC=0.744

--- Running k=40 ---
k=40: RF macro-AUC=0.829 | ElasticNet macro-AUC=0.852

    n_features         model  macro_auc
0           5  RandomForest   0.714915
1           5    ElasticNet   0.734732
2          10  RandomForest   0.744951
3          10    ElasticNet   0.751177
4          20  RandomForest   0.760445
5          20    ElasticNet   0.743626
6          40  RandomForest   0.828584
7          40    ElasticNet   0.852189
